# 08 — Feature selection (gated S4 pipeline, protocol v3)

"Compute wide, select narrow": 35 candidates → 5–8 for the final model.
Steps (each validated before the next): **② de-redundancy** (here) → ③ label → ③bis regime
probabilities → ④ XGBoost importance selection → ⑤ central test VIX vs VIX+spectral.

**② De-redundancy — no label, no model.** Feature-feature correlation matrix
(Spearman, robust to fat tails), clusters of |ρ| > 0.8, keep the most interpretable per cluster.
*Look-ahead: none by construction* (feature-feature does not touch the target); we additionally
check the **stability** of the structure over the two halves of the sample.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
df = pd.read_parquet(ROOT / "data" / "processed" / "ml_dataset.parquet")
feats = [c for c in df.columns if c.startswith("f_")]
print(f"{len(feats)} candidate features | {df.shape[0]} days")
X = df[feats]
cov = X.notna().mean().sort_values()
print("\ncoverage (5 lowest):")
print((cov.head() * 100).round(1).to_string())

38 candidate features | 7221 days

coverage (5 lowest):
f_regime         85.9
f_reg_gmm        85.9
f_reg_hmm        85.9
f_sig_rmt_pct    95.3
f_dlam1_63       96.8


In [2]:
# Pairwise Spearman correlation (complete-observations per pair)
C = X.corr(method="spearman")

# Stability check: |rho| of strong pairs on 1st vs 2nd half
half = len(df) // 2
C1 = X.iloc[:half].corr(method="spearman").abs()
C2 = X.iloc[half:].corr(method="spearman").abs()
iu = np.triu_indices(len(feats), 1)
strong = np.abs(C.values[iu]) > 0.8
pairs = [(feats[i], feats[j], C.values[i, j], C1.values[i, j], C2.values[i, j])
         for i, j, s in zip(iu[0], iu[1], strong) if s]
sp = pd.DataFrame(pairs, columns=["f_a", "f_b", "rho_full", "|rho|_1H", "|rho|_2H"])
sp = sp.reindex(sp["rho_full"].abs().sort_values(ascending=False).index)
print(f"pairs |rho| > 0.8: {len(sp)}")
print("stability: min(|rho|) over the two halves among these pairs =",
      round(sp[["|rho|_1H", "|rho|_2H"]].min().min(), 2), "(→ stable structure if high)")
sp.round(3).to_string(index=False)

pairs |rho| > 0.8: 22
stability: min(|rho|) over the two halves among these pairs = 0.68 (→ stable structure if high)


'      f_a            f_b  rho_full  |rho|_1H  |rho|_2H\n    f_abs      f_entropy    -0.984     0.983     0.964\n   f_lam1 f_rho_clean252     0.981     0.984     0.965\n f_iv_spx          f_vix     0.978     0.983     0.963\nf_entropy f_rho_clean252    -0.966     0.970     0.976\n   f_lam1      f_entropy    -0.964     0.977     0.955\nf_sig_rmt  f_sig_rmt_pct     0.949     0.900     0.995\nf_reg_hmm       f_regime     0.928     0.970     0.880\n    f_abs f_rho_clean252     0.925     0.940     0.919\n   f_lam1         f_prv1     0.913     0.869     0.944\n   f_lam1          f_abs     0.910     0.939     0.862\n    f_vix      f_iv_comp     0.898     0.876     0.840\n f_iv_spx      f_iv_comp     0.886     0.871     0.779\n   f_prv1 f_rho_clean252     0.879     0.831     0.893\nf_iv_comp     f_iv_xdisp     0.871     0.870     0.838\n f_iv_spx         f_rv63     0.854     0.874     0.773\nf_entropy         f_prv1    -0.853     0.803     0.888\n   f_rv21         f_rv63     0.847     0.867   

In [3]:
# Hierarchical clustering: distance = 1 - |rho|, average linkage, cut at |rho| = 0.8
D = 1.0 - C.abs().values
np.fill_diagonal(D, 0.0)
D = np.nan_to_num(D, nan=1.0)                    # pairs without overlap -> uncorrelated
D = (D + D.T) / 2
Z = linkage(squareform(D, checks=False), method="average")
labels = fcluster(Z, t=0.2, criterion="distance")   # 1 - 0.8
clusters = {}
for f, l in zip(feats, labels):
    clusters.setdefault(l, []).append(f)
multi = {k: v for k, v in clusters.items() if len(v) > 1}
singles = [v[0] for v in clusters.values() if len(v) == 1]
print(f"{len(clusters)} clusters: {len(multi)} redundant, {len(singles)} isolated features\n")
for k, v in sorted(multi.items(), key=lambda kv: -len(kv[1])):
    print(f"redundant cluster ({len(v)}): {v}")
print(f"\nisolated features (kept by default): {sorted(singles)}")

28 clusters: 4 redundant, 24 isolated features

redundant cluster (5): ['f_lam1', 'f_abs', 'f_entropy', 'f_prv1', 'f_rho_clean252']
redundant cluster (5): ['f_iv_spx', 'f_vix', 'f_rv21', 'f_rv63', 'f_iv_comp']
redundant cluster (2): ['f_sig_rmt', 'f_sig_rmt_pct']
redundant cluster (2): ['f_reg_hmm', 'f_regime']

isolated features (kept by default): ['f_anchor_gap', 'f_cost_era', 'f_dd252', 'f_div_21', 'f_dlam1_21', 'f_dlam1_63', 'f_drho_21', 'f_drho_imp_21', 'f_iv_xdisp', 'f_k', 'f_lam2', 'f_mom63', 'f_reg_gmm', 'f_rho_imp', 'f_rho_trail63', 'f_rot21', 'f_sig_base', 'f_skew50', 'f_term_slope', 'f_turb', 'f_turb21', 'f_volofvol', 'f_vrp', 'f_xdisp']


In [4]:
# DECISION ② (pre-fixed rule: the most interpretable/standard, direct economic level > derived)
import json

KEEP = {
    "market-mode strength block": dict(
        survivant="f_lam1",
        eliminees={"f_abs": "corr 0.91 with λ₁/N (top-5 ≈ λ₁ + noise)",
                   "f_entropy": "corr −0.96: entropy IS (1−λ₁/N) reparameterised",
                   "f_prv1": "corr 0.91: participation ≈ width of the same mode",
                   "f_rho_clean252": "corr 0.98: the slow anchor ≈ market-mode share — info retained via f_anchor_gap (isolated)"},
        justification="λ₁/N = THE canonical RMT quantity (Laloux/BBP), direct thesis narrative, user HMM spec",
    ),
    "vol level block": dict(
        survivant="f_vix",
        eliminees={"f_iv_spx": "corr 0.978 with VIX (91d ATM vs 30d MFIV: same level factor)",
                   "f_iv_comp": "corr 0.90/0.89 with VIX/IV-SPX",
                   "f_rv21": "corr 0.84: RV tracks the level; the useful gap is already f_vrp (isolated)",
                   "f_rv63": "corr 0.83-0.85: same"},
        justification="the official VIX = required by the CENTRAL TEST \"VIX only\" (literature credibility); f_vrp keeps the IV−RV gap",
    ),
    "rmt signal block": dict(
        survivant="f_sig_rmt",
        eliminees={"f_sig_rmt_pct": "corr 0.95: expanding rank = monotone transform of the signal once history is long"},
        justification="direct economic level (the input of the v1_rmt gate) > normalised derivative",
    ),
}

eliminated = [f for v in KEEP.values() for f in v["eliminees"]]
survivors_clusters = [v["survivant"] for v in KEEP.values()]
candidates = sorted(set(feats) - set(eliminated))
assert set(survivors_clusters) <= set(candidates) and len(candidates) == len(feats) - len(eliminated)
print(f"35 candidates -> {len(eliminated)} eliminated -> {len(candidates)} retained for step ④:\n")
for k, v in KEEP.items():
    print(f"[{k}] survivor: {v['survivant']}  — {v['justification']}")
    for f, why in v["eliminees"].items():
        print(f"   ✗ {f}: {why}")
print(f"\npost-de-redundancy list ({len(candidates)}): {candidates}")

with open(ROOT / "data" / "processed" / "ml_candidates_postdedup.json", "w") as fh:
    json.dump({"candidates": candidates, "eliminated": eliminated,
               "rule": "clusters |rho|>0.8, most interpretable survivor (notebook 08, step 2)"}, fh, indent=2)
print("\n→ data/processed/ml_candidates_postdedup.json written")

35 candidates -> 9 eliminated -> 29 retained for step ④:

[market-mode strength block] survivor: f_lam1  — λ₁/N = THE canonical RMT quantity (Laloux/BBP), direct thesis narrative, user HMM spec
   ✗ f_abs: corr 0.91 with λ₁/N (top-5 ≈ λ₁ + noise)
   ✗ f_entropy: corr −0.96: entropy IS (1−λ₁/N) reparameterised
   ✗ f_prv1: corr 0.91: participation ≈ width of the same mode
   ✗ f_rho_clean252: corr 0.98: the slow anchor ≈ market-mode share — info retained via f_anchor_gap (isolated)
[vol level block] survivor: f_vix  — the official VIX = required by the CENTRAL TEST "VIX only" (literature credibility); f_vrp keeps the IV−RV gap
   ✗ f_iv_spx: corr 0.978 with VIX (91d ATM vs 30d MFIV: same level factor)
   ✗ f_iv_comp: corr 0.90/0.89 with VIX/IV-SPX
   ✗ f_rv21: corr 0.84: RV tracks the level; the useful gap is already f_vrp (isolated)
   ✗ f_rv63: corr 0.83-0.85: same
[rmt signal block] survivor: f_sig_rmt  — direct economic level (the input of the v1_rmt gate) > normalised derivative
  

In [5]:
# NARROWING TO 10 (economic judgment, authorised by the user — more robust than
# in-sample importance over 12 crises, which would overfit the feature choice).
# HMM/GMM regime prob ADDED ON TOP no matter what (11th input) — user directive.
# f_cost_era REMOVED from predictors -> reserved for the cost-aware GATE (full-sample calibration caveat).
FINAL10 = {
    # --- vol baseline ("VIX only" arm of the central test) ---
    "f_vix":          "official vol level — literature baseline of the central test",
    "f_term_slope":   "term structure 30->91d — the most documented stress indicator",
    # --- spectral RMT (the contribution TESTED beyond the VIX) ---
    "f_lam1":         "λ₁/N, market-mode strength — canonical RMT quantity",
    "f_dlam1_63":     "λ₁/N dynamics over 63d — the correlation that RISES (precursor, horizon-consistent)",
    "f_turb21":       "Mahalanobis turbulence on the CLEANED inverse — where RMT clipping has value",
    "f_k":            "number of factors > Laloux edge — regime width (distinct from the λ₁ level)",
    "f_rot21":        "rotation of the dominant eigenvector — regime CHANGE (orthogonal to the level)",
    # --- economic / signal ---
    "f_vrp":          "variance risk premium IV−RV — documented crash predictor",
    "f_drho_imp_21":  "change in IMPLIED correlation — the market STARTS to price stress",
    "f_sig_rmt":      "the primary signal itself — meta-labeling context (strength of the proposed bet)",
}
final10 = list(FINAL10)
assert len(final10) == 10 and set(final10) <= set(candidates)
core_hmm = ["f_lam1", "f_dlam1_63", "f_turb21", "f_vix"]   # 3-4 features for HMM/GMM (v3 spec)
assert set(core_hmm) <= set(final10)

print("10 final features (judgment, label-free):")
for f, why in FINAL10.items():
    print(f"  {f:16s} {why}")
print(f"\n+ HMM/GMM regime prob (stacking, on top) = 11 inputs")
print(f"core set HMM/GMM (3-4): {core_hmm}")
print(f"f_cost_era -> cost-aware gate only (outside prediction)")

with open(ROOT / "data" / "processed" / "ml_features_final.json", "w") as fh:
    json.dump({"final10": final10, "core_hmm": core_hmm,
               "cost_gate_only": "f_cost_era",
               "rule": "economic judgment + block diversity (notebook 08, step 2->10)"}, fh, indent=2)
print("\n→ data/processed/ml_features_final.json written")

10 final features (judgment, label-free):
  f_vix            official vol level — literature baseline of the central test
  f_term_slope     term structure 30->91d — the most documented stress indicator
  f_lam1           λ₁/N, market-mode strength — canonical RMT quantity
  f_dlam1_63       λ₁/N dynamics over 63d — the correlation that RISES (precursor, horizon-consistent)
  f_turb21         Mahalanobis turbulence on the CLEANED inverse — where RMT clipping has value
  f_k              number of factors > Laloux edge — regime width (distinct from the λ₁ level)
  f_rot21          rotation of the dominant eigenvector — regime CHANGE (orthogonal to the level)
  f_vrp            variance risk premium IV−RV — documented crash predictor
  f_drho_imp_21    change in IMPLIED correlation — the market STARTS to price stress
  f_sig_rmt        the primary signal itself — meta-labeling context (strength of the proposed bet)

+ HMM/GMM regime prob (stacking, on top) = 11 inputs
core set HMM/GMM (3